In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import os
import json

from dotenv import load_dotenv
load_dotenv()

import ollama

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langgraph.prebuilt import InjectedState

from file_tools import (
    DeepAgentState,
    ls,
    read_file,
    write_file,
    cleanup_files,
)
from prompts import (
    ORCHESTRATOR_PROMPT,
    RESEARCHER_PROMPT,
    EDITOR_PROMPT,
)

In [3]:
# -------------------------
# Web Search Tool
# -------------------------

@tool
def web_search(query: str) -> str:
    """
    Perform a live web search using Ollama Cloud Web Search API.

    Input:
        query: search query string

    Output:
        JSON string of top results (max_results=2).
    """
    response = ollama.web_search(query, max_results=2)
    response = response.results

    return response


In [4]:
# result = web_search.invoke({"query": "Latest global news"})

In [5]:
# -------------------------
# LLM
# -------------------------

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

In [6]:
# -------------------------
# Worker Agents: Researcher & Editor
# -------------------------

researcher_tools = [ls, write_file, read_file, web_search]
researcher_agent = create_agent(
    model=llm,
    tools=researcher_tools,
    system_prompt=RESEARCHER_PROMPT,
    state_schema=DeepAgentState,
)

editor_tools = [ls, read_file, write_file, cleanup_files]
editor_agent = create_agent(
    model=llm,
    tools=editor_tools,
    system_prompt=EDITOR_PROMPT,
    state_schema=DeepAgentState,
)


In [7]:
# -------------------------
# Orchestrator Tools (call other agents)
# -------------------------
from typing import Annotated
from langgraph.prebuilt import InjectedState
from langchain_core.tools import InjectedToolCallId
from langgraph.types import Command

@tool
def run_researcher(
    state: Annotated[DeepAgentState, InjectedState]
) -> str:
    """
    Run the Research agent for this user/thread using the current conversation.

    The Researcher will:
    - read research_plan.md to see thematic questions
    - break each theme into focused search queries
    - use web_search internally
    - write theme_1.md, theme_2.md, etc. and sources.txt

    Returns a short status string for the orchestrator.
    """
    sub_state: DeepAgentState = {
        "messages": state["messages"],
        "user_id": state.get("user_id"),
        "thread_id": state.get("thread_id"),
    }
    researcher_agent.invoke(sub_state)
    return "Research completed. Theme files and sources.txt are updated."


In [8]:
@tool
def write_research_plan(
    thematic_questions: list[str],
    state: Annotated[DeepAgentState, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> Command:
    """
    Write the high-level research plan with major thematic questions.

    Args:
        thematic_questions: List of 3-5 major thematic questions that break down
                           the user's query. These guide the Researcher agent.
        state: Injected agent state providing user_id/thread_id.
        tool_call_id: Tool call ID for attaching a ToolMessage.

    The research_plan.md will be read by the Researcher to guide tactical research.

    Returns:
        Command with a ToolMessage confirming the plan was written.
    """
    from file_tools import _disk_path
    from langchain_core.messages import ToolMessage
    from langgraph.types import Command
    
    # Build content for research_plan.md
    content = "# Research Plan\n\n"
    content += "## User Query\n"
    # Extract user query from messages
    user_msg = [m for m in state["messages"] if hasattr(m, 'content')]
    if user_msg:
        content += f"{user_msg[-1].content}\n\n"
    
    content += "## Thematic Questions\n\n"
    for i, question in enumerate(thematic_questions, 1):
        content += f"{i}. {question}\n"
    
    # Write to disk
    path = _disk_path(state, "research_plan.md")
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    
    msg = f"[RESEARCH PLAN WRITTEN] research_plan.md with {len(thematic_questions)} thematic questions -> {path}"
    return Command(
        update={
            "messages": [ToolMessage(msg, tool_call_id=tool_call_id)]
        }
    )


@tool
def run_editor(
    state: Annotated[DeepAgentState, InjectedState]
) -> str:
    """
    Run the Editor agent for this user/thread.

    The Editor will:
    - read research_plan.md, theme files, sources.txt
    - write the final answer to report.md

    Returns a short status string for the orchestrator.
    """
    sub_state: DeepAgentState = {
        "messages": state["messages"],
        "user_id": state.get("user_id"),
        "thread_id": state.get("thread_id"),
    }
    editor_agent.invoke(sub_state)
    return "Editor completed. Final report is written to report.md."

In [9]:
# -------------------------
# Orchestrator Agent
# -------------------------

orchestrator_tools = [
    write_research_plan,  # strategic planning
    run_researcher,       # tactical research
    run_editor,           # synthesis
    cleanup_files,        # resetting workspace
]

orchestrator_agent = create_agent(
    model=llm,
    tools=orchestrator_tools,
    system_prompt=ORCHESTRATOR_PROMPT,
    state_schema=DeepAgentState,
)


### Example usage


In [10]:
from langchain.messages import HumanMessage
state = {
        "messages": [HumanMessage(content="Tell me a joke about computers.")],
        "user_id": "user_123",
        "thread_id": "thread_mcp_001",
    }

result = orchestrator_agent.invoke(state)
result

{'messages': [HumanMessage(content='Tell me a joke about computers.', additional_kwargs={}, response_metadata={}, id='752e51fd-73a5-444e-b114-c93583a6a85a'),
  AIMessage(content='Why was the computer cold? \nBecause it left its Windows open!', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--885c68b0-8528-4f62-af4e-b4f495acb907-0', usage_metadata={'input_tokens': 1644, 'output_tokens': 15, 'total_tokens': 1659, 'input_token_details': {'cache_read': 0}})],
 'user_id': 'user_123',
 'thread_id': 'thread_mcp_001'}

In [ ]:
from langchain.messages import HumanMessage
state = {
        "messages": [HumanMessage(content="Do a detailed, well-structured analysis of MCP including history")],
        "user_id": "user_123",
        "thread_id": "thread_mcp_001",
    }

result = orchestrator_agent.invoke(state)

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 250000, model: gemini-2.5-flash
Please retry in 35.220553524s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_input_token_count"
  quota_id: "GenerateContentInputTokensPerModelPerMinute-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global

In [12]:
result

{'messages': [HumanMessage(content='Do a detailed, well-structured analysis of MCP including history', additional_kwargs={}, response_metadata={}, id='ba2dfb13-c4f7-4faa-8fe3-d817cb4028aa'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'write_research_plan', 'arguments': '{"thematic_questions": ["What is MCP and what problem does it solve?", "What is the history and evolution of MCP?", "What are the key architectural components and principles of MCP?", "What are practical use cases and applications of MCP?", "What are the advantages and limitations of MCP?"]}'}, '__gemini_function_call_thought_signatures__': {'caaaaa6d-b59a-4c7c-ad44-097632efaf15': 'CrQHAXLI2nw5qyaKehi+XxPf/qE47B8DrrDVi2383irBuOXaoYqcF8G95rRqBUG5tvG2mSt1z5zZZ2JAYFkDI9vQeTStgrI7WfXthmwSiLYbKcEEwls4C6WEgAAyLD5TZy/DdpUJlJzDSVzve9oDhktu0ZlBobggLkK8nH+NTXlptRfr1X0rBA1dBn/3gx/oYRLWpJegH2J9HouNs7Xo/16ms6mikutIuXOll6dwf1F9XZB5XyDyvL9gIN5LGjoDW0qN16R4x0kjZwfrXZ15nP24qnymEop920RK2tux4fGB5pgtVw2IiP/aLM9yib

In [13]:
from langchain.messages import HumanMessage
state = {
        "messages": [HumanMessage(content="Do a detailed, well-structured comparison of MCP and API.")],
        "user_id": "user_123",
        "thread_id": "thread_mcp_001",
    }

result = orchestrator_agent.invoke(state)

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 10, model: gemini-2.5-flash
Please retry in 48.076131207s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 1

In [14]:
result

{'messages': [HumanMessage(content='Do a detailed, well-structured comparison of MCP and API.', additional_kwargs={}, response_metadata={}, id='9d84174a-4aee-400a-8123-c3833987ade5'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'write_research_plan', 'arguments': '{"thematic_questions": ["What are the definitions and core functionalities of MCP (Master Control Program) and API (Application Programming Interface)?", "What are the key architectural differences and similarities between MCP and API?", "What are the primary use cases and application scenarios for both MCP and API?", "What are the comparative advantages and disadvantages of MCP and API in different contexts?"]}'}, '__gemini_function_call_thought_signatures__': {'6d6364a7-cae0-4f55-af32-e7fbc6297f3f': 'CtkLAXLI2ny6LvldGmGoH6asK/Yp/vCiHtRfgRoCyqqaKYYiETiTluX0xxrdqj+uW9fAa0evPG5lMZhJKPobTs72V8BC5PqqI2POsITlNPQU4z7tluR7HqOcbmbstFL9kSkfn/Zxvy0ecKsPyrwOajyfzJYFiT8QF2JX1CsQocnw7LFsZ2IUe4IIzfwHnJmLlLPcnz5+rx

In [15]:
# -------------------------
# Test: Verify Enhanced Hierarchical Workflow
# -------------------------

print("=" * 60)
print("ENHANCED MULTI-AGENT HIERARCHICAL RESEARCH SYSTEM")
print("=" * 60)

print("\n[ORCHESTRATOR] - Strategic Planner & Coordinator")
print("Tools:")
for tool in orchestrator_tools:
    print(f"  - {tool.name}")

print("\n[RESEARCHER] - Tactical Researcher & Information Gatherer")
print("Tools:")
for tool in researcher_tools:
    print(f"  - {tool.name}")

print("\n[EDITOR] - Synthesis Specialist & Report Writer")
print("Tools:")
for tool in editor_tools:
    print(f"  - {tool.name}")

print("\n" + "=" * 60)
print("WORKFLOW HIERARCHY")
print("=" * 60)
print("""
1. USER QUERY
   ↓
2. ORCHESTRATOR: Strategic Planning
   - Analyzes user query
   - Breaks it into 3-5 major thematic questions
   - Calls write_research_plan(thematic_questions=[...])
   ↓
3. ORCHESTRATOR: Delegates to Researcher
   - Calls run_researcher()
   ↓
4. RESEARCHER: Tactical Research
   - Reads research_plan.md
   - For each thematic question:
     * Breaks it into 2-4 focused search queries
     * Performs web_search() for each query
     * Writes theme_N.md with findings
   - Compiles sources.txt
   ↓
5. ORCHESTRATOR: Delegates to Editor
   - Calls run_editor()
   ↓
6. EDITOR: Synthesis & Report Generation
   - Reads research_plan.md (structure)
   - Reads all theme_N.md files (content)
   - Reads sources.txt (references)
   - Synthesizes into final report.md
   ↓
7. ORCHESTRATOR: Informs user
   - Report complete, saved to report.md
""")

ENHANCED MULTI-AGENT HIERARCHICAL RESEARCH SYSTEM

[ORCHESTRATOR] - Strategic Planner & Coordinator
Tools:
  - write_research_plan
  - run_researcher
  - run_editor
  - cleanup_files

[RESEARCHER] - Tactical Researcher & Information Gatherer
Tools:
  - ls
  - write_file
  - read_file
  - web_search

[EDITOR] - Synthesis Specialist & Report Writer
Tools:
  - ls
  - read_file
  - write_file
  - cleanup_files

WORKFLOW HIERARCHY

1. USER QUERY
   ↓
2. ORCHESTRATOR: Strategic Planning
   - Analyzes user query
   - Breaks it into 3-5 major thematic questions
   - Calls write_research_plan(thematic_questions=[...])
   ↓
3. ORCHESTRATOR: Delegates to Researcher
   - Calls run_researcher()
   ↓
4. RESEARCHER: Tactical Research
   - Reads research_plan.md
   - For each thematic question:
     * Breaks it into 2-4 focused search queries
     * Performs web_search() for each query
     * Writes theme_N.md with findings
   - Compiles sources.txt
   ↓
5. ORCHESTRATOR: Delegates to Editor
   - Calls